# Native-detail H/L — một adapter chung, T4 ×2
**Một notebook, hai GPU**, không chạy hai mô hình H/L độc lập.
Exact A frozen;262.656 tham số adapter;H/L cùng trọng số. H/L→Q' rồi tính lại legacy grounding→decoder A.
**CE-only**, đủ5caption TRAIN có EOS,4epoch/5.000update,globalbatch8,seed42.
Không thêm SCST/object reward, không chọn checkpoint theo val. Kiến thức SCST trước đây vẫn nằm trong A.

Chọn **T4 ×2**, Internet ON, **Save & Run All**. Root không tự chạy notebook.
Thêm5dataset: **native-detail-hl-15k-datase**, **pair-source-fullframe-ce**, **prompt-cache**, **visual-cache**, **mscoco-2014**.
Không cần ảnh gốc/detectioncache/UPT/VG cache mới.

Notebook kiểm hash cache thật, stage nguyên dữ liệu A (không lượng tử hóa thêm), kiểm zero-init/gradient,
đo25bước CE và tốc độ sinh trên mẫu TRAIN rồi khôi phục head+optimizer+seed ban đầu.
Nếu ước lượng có hệ số dự phòng1,5 trong ngân sách **8giờ wall**, notebook tiếp tục5.000update và đánh giá.
Nếu không đủ ngân sách hoặc preflight lỗi, dừng với review ZIP; không gọi lượt thiếu là kết quả âm.
Bạn báo còn30giờGPU;ước lượng wall không được coi là số giờ quota thực tế Kaggle sẽ tính.

Đánh giá rawgreedy A/L/H trên **reusedval5000**, không test. Gate đầu tư:H−A≥.01,H−L≥.005,cả haiCIcận dưới>0,
CHAIRs/i≤A vàRecall≥A−.005. Không tự scale/quét hệ số nếu fail.
Gửi `native_detail_shared_HL_pilot_review.zip`; giữ `head.bin` **và checkpoint A** để dùng mô hình.

In [ ]:
print('SHARED H/L PILOT: START',flush=True)
from pathlib import Path
HL_CACHE=Path('/kaggle/input/datasets/ducanh2403/native-detail-hl-15k-datase/native_detail_HL_15k_dataset')
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
OUTPUT=Path('/kaggle/working/native_detail_shared_HL_pilot')
STAGE=Path('/tmp/native_detail_shared_HL_stage.h5')
for p in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON,HL_CACHE/'cache_spec.json',HL_CACHE/'cache_receipt.json'):assert p.is_file(),p
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists() and not STAGE.exists(),'Preserve existing output/stage; use a fresh session or paths.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-native-detail-shared-pilot-'))/'repo'
LOG = Path('/tmp/upt_hidden_native-detail-shared-pilot.log')
PROJECT_COMMIT = '8b8516959dba8e8519ba6536c9169b2469e3506b'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'detail_data.py': '4b969722e8f5efce5c77b8357723510239a494d0974fe22f4fe6ee50f4d22d5d', 'detail_model.py': 'f18cb1b6cb3a767a2217a9bcf8d107a18683562ec8236d0dca629fe4e81ee7d4', 'detail_stage.py': 'ca93c4b19ad2f0655169fed4ce5b8ad73cb4042a25097a0fbf83c87b01174b17', 'run_detail.py': '43e7ffbf3079b4d8f6afa96f31525b26c6a4e393440aa1a25d7541075a31b9d9', 'detail_metrics.py': 'efc3cf631de5ab1b7a6bf46b2846bcdacd37f562e044d936bd666834a8907c8c', 'test_detail.py': '2087fcc9ca7eec2b344f2724c425bcf28a291f46bb9efd9a647e647797aee35d', 'tile_data.py': '47059a7eeca1b7fd3edfa7bfe2a52bbab4ea6b506bcf322bb6962d163c1512be', 'hl_upload_inventory.json': '1eb10f48ba842bc4ead2af9d68cf42617daca1678a454a818183f88b98fae36d', 'hidden_metric_smoke.py': '8cf64c56b169cdd2407fdeb668408a80d947578632154405c47c7040558a698b', 'captioning/models.py': '18026877836052c83e75c0d896adcc89822bb35616173c7e8b14b17c9a22e9d8', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'captioning/tokenizer.py': 'df2ace0025aa7d4e12c289ab7bca68046b48d4b6c2b5c12606a19ff5a50a638e', 'captioning/config.py': '4766d1bf82bf82b5199616c7d90ca74242f011e26f08db21b154b1c7d58e72bc', 'captioning/semantic_prompts.py': 'a06f12dea63604f9db9d19b6922815db27a35b648a17059407ffaf26a2a0d511', 'captioning/visual_cache.py': 'adbcc9678fff499fc767cdbf04a4966186a40729f51c5bfb2525e79f9b6b5fe1'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','numpy','Pillow','pandas',
    'pycocotools','pycocoevalcap==1.2','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])
import torch
assert torch.cuda.device_count()==2 and all('T4' in torch.cuda.get_device_name(r) for r in range(2)),'Select T4x2.'
run([sys.executable,'-u','test_detail.py'],cwd=REPO)
CHAIR=Path('/tmp/native-detail-pilot-chair')
checkout('https://github.com/Maxlinn/CHAIR-metric-standalone.git',CHAIR,'4087a26211aa2339b9a76307cb8f0321ef691d0a')
run([sys.executable,'-u','hidden_metric_smoke.py',CHAIR],cwd=REPO)


In [ ]:
CONFIG=Path('/tmp/native_detail_pilot_config.json')
cfg=dict(checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),dataset_json=str(COCO_JSON),
    tile_cache=str(HL_CACHE),output=str(OUTPUT),stage=str(STAGE),inventory=str(REPO/'hl_upload_inventory.json'),chair=str(CHAIR),
    epochs=4,updates=5000,global_batch_size=8,eval_batch_size=16,seed=42,head_lr=1e-4,weight_decay=.01,clip_norm=1.,head_parameters=262656,
    wall_budget_seconds=8*3600,reported_gpu_quota_hours=30,project_commit=PROJECT_COMMIT,implementation_sha256=SOURCE_HASHES,
    target_policy='Five natural TRAIN captions per image,truncatewithEOS;BOSexcluded/PADignored/globaltokenCE includingEOS',
    evaluation='Endpoint only;rawgreedy,maxlen30,A/L/H,reusedval5000;notest/norerank/nosweep',
    planned_gate=dict(H_A_CIDEr=.01,H_L_CIDEr=.005,paired95lower_gt=0,CHAIRs_i_not_above_A=True,Recall_drop_max=.005))
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8');PROTOCOL=OUTPUT/'protocol.json'
try:
    run([sys.executable,'-u','run_detail.py','prepare','--config',CONFIG],cwd=REPO)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',
        'run_detail.py','worker','--config',PROTOCOL],cwd=REPO)
    run([sys.executable,'-u','run_detail.py','evaluate','--config',PROTOCOL],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        if LOG.is_file():shutil.copyfile(LOG,OUTPUT/'notebook.log')
        if not PROTOCOL.is_file():shutil.copyfile(CONFIG,OUTPUT/'requested_config.json')
        for name,source in SOURCES.items():
            target=OUTPUT/'source'/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_text(source,encoding='utf-8')
        run([sys.executable,'-u','run_detail.py','archive','--config',PROTOCOL if PROTOCOL.is_file() else CONFIG],cwd=REPO)
summary=json.loads((OUTPUT/'run_summary.json').read_text(encoding='utf-8'))
assert summary['status']=='PILOT_COMPLETE','ZIP alone is not success; send incomplete review for diagnosis.'
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP ADAPTER:',OUTPUT/'head.bin','KEEP ORIGINAL A CHECKPOINT SEPARATELY',flush=True)
